# 00 — Verificación del Entorno

Este notebook configura la API de Kaggle, descarga el dataset **Landslide4Sense** y verifica la integridad de los 14 canales multi-espectrales.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

## 1. CONFIGURACIÓN DE COLAB Y KAGGLE
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    print("📦 Entorno Colab detectado.")
    subprocess.run([sys.executable, "-m", "pip", "install", "kaggle", "--upgrade", "-q"], check=True)

    from google.colab import files
    kaggle_path = Path('/root/.kaggle/kaggle.json')
    if not kaggle_path.exists():
        print("🔑 Sube tu archivo kaggle.json:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            kaggle_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.move('kaggle.json', str(kaggle_path))
            kaggle_path.chmod(0o600)
            print("✓ Credenciales configuradas.")
    
    print("🛰️ Descargando dataset...")
    try:
        subprocess.run(["kaggle", "datasets", "download", "-d", "lxrzp/landslide4sense", "-p", "/content/", "--unzip"], check=True)
        print("✓ Descarga exitosa.")
    except Exception as e:
        print(f"❌ Error: {e}")

    for _root, _dirs, _ in os.walk('/content'):
        if 'TrainData' in _dirs:
            DATA_ROOT_OVERRIDE = _root
            break
else:
    print("💻 Entorno local detectado.")

In [ ]:
## 2. VERIFICACIÓN DE DATOS Y ESTRUCTURA
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
else:
    _cwd = Path.cwd()
    _candidates = [_cwd / ".." / "data", _cwd / "data", _cwd.parent]
    DATA_ROOT = next((c.resolve() for c in _candidates if (c / "TrainData" / "img").exists()), None)

if DATA_ROOT:
    print(f"📂 DATA_ROOT: {DATA_ROOT}")
    for part in ["TrainData", "ValidData", "TestData"]:
        img_p = DATA_ROOT / part / "img"
        n = len(list(img_p.glob("*.h5"))) if img_p.exists() else 0
        print(f"  {'✓' if n > 0 else '✗'} {part}: {n} archivos .h5")
else:
    print("❌ No se encontró TrainData.")

In [ ]:
## 3. INSTALACIÓN DE LIBRERÍAS Y TEST DE CARGA
if IN_COLAB:
    !pip install -q segmentation-models-pytorch timm h5py

import h5py
import matplotlib.pyplot as plt

img_list = sorted((DATA_ROOT / "TrainData" / "img").glob("*.h5"))
if img_list:
    with h5py.File(img_list[0], 'r') as f:
        data = f['img'][:]
    print(f"✓ Patch cargado. Shape: {data.shape}")
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, ch in enumerate([0, 7, 9]):
        axes[i].imshow(data[:,:,ch], cmap='gray')
        axes[i].set_title(f"Canal {ch}")
        axes[i].axis('off')
    plt.show()